# 📊 Student Exam Score Dashboard
**Dataset:** `17_student_exam_scores.csv` | 200 Students | 6 Features  
**Goal:** Explore unique data insights through an interactive visual dashboard


## 📦 Install & Import Libraries

In [ ]:
# Run once if needed:
# !pip install pandas numpy matplotlib seaborn plotly ipywidgets

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style="whitegrid", palette="muted")
print("✅ Libraries ready")


## Step 1 — Extract Unique & Meaningful Data
We derive **new insight columns** from the raw data to make the dashboard richer:
- `performance_grade` — letter grade from exam_score
- `study_category` — Low / Medium / High study hours
- `sleep_quality` — Poor / Fair / Good sleep
- `pass_fail` — Pass (≥30) or Fail
- `improvement_gap` — exam_score minus previous_scores


In [ ]:
df = pd.read_csv("17_student_exam_scores.csv")

# ── Derived Unique Columns ──────────────────────────────────
df['performance_grade'] = pd.cut(
    df['exam_score'],
    bins=[0, 25, 30, 35, 40, 100],
    labels=['F (<25)', 'D (25-30)', 'C (30-35)', 'B (35-40)', 'A (40+)']
)

df['study_category'] = pd.cut(
    df['hours_studied'],
    bins=[0, 4, 8, 13],
    labels=['Low (0-4h)', 'Medium (4-8h)', 'High (8+h)']
)

df['sleep_quality'] = pd.cut(
    df['sleep_hours'],
    bins=[0, 5.5, 7.5, 10],
    labels=['Poor (<5.5h)', 'Fair (5.5-7.5h)', 'Good (7.5+h)']
)

df['pass_fail'] = df['exam_score'].apply(lambda x: 'Pass ✅' if x >= 30 else 'Fail ❌')

df['improvement_gap'] = df['exam_score'] - df['previous_scores']

df['attendance_band'] = pd.cut(
    df['attendance_percent'],
    bins=[0, 60, 75, 90, 101],
    labels=['Low (<60%)', 'Average (60-75%)', 'Good (75-90%)', 'Excellent (90%+)']
)

print("✅ Unique columns added:", ['performance_grade','study_category',
      'sleep_quality','pass_fail','improvement_gap','attendance_band'])
print(f"\n📌 Dataset shape: {df.shape}")
df.head()


In [ ]:
# ── Unique value counts summary ────────────────────────────
print("=" * 50)
print("  UNIQUE VALUE COUNTS PER DERIVED COLUMN")
print("=" * 50)
for col in ['performance_grade','study_category','sleep_quality','pass_fail','attendance_band']:
    print(f"\n📌 {col}:")
    print(df[col].value_counts().to_string())


## Step 2 — Full Dashboard (Jupyter)
A **6-panel dashboard** covering grades, study habits, sleep, attendance, correlation, and improvement.


In [ ]:
fig = plt.figure(figsize=(20, 22))
fig.patch.set_facecolor('#0f1117')
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

PANEL_BG  = '#1e2130'
TEXT_COL  = '#e8eaf6'
GRID_COL  = '#2e3250'

PALETTE   = ['#4fc3f7','#81c784','#ffb74d','#e57373','#ce93d8']

def style_ax(ax, title):
    ax.set_facecolor(PANEL_BG)
    ax.tick_params(colors=TEXT_COL, labelsize=9)
    ax.set_title(title, color=TEXT_COL, fontsize=11, fontweight='bold', pad=10)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID_COL)
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.grid(color=GRID_COL, linestyle='--', linewidth=0.5, alpha=0.6)

# ── Panel 1: Grade Distribution (Pie) ──────────────────────
ax1 = fig.add_subplot(gs[0, 0])
grade_counts = df['performance_grade'].value_counts().sort_index()
wedges, texts, autotexts = ax1.pie(
    grade_counts, labels=grade_counts.index,
    autopct='%1.1f%%', colors=PALETTE,
    startangle=140, textprops={'color': TEXT_COL, 'fontsize': 8},
    wedgeprops={'edgecolor': '#0f1117', 'linewidth': 1.5}
)
for at in autotexts: at.set_fontsize(8)
ax1.set_facecolor(PANEL_BG)
ax1.set_title("📊 Grade Distribution", color=TEXT_COL, fontsize=11, fontweight='bold')

# ── Panel 2: Study Category vs Avg Exam Score (Bar) ────────
ax2 = fig.add_subplot(gs[0, 1])
study_avg = df.groupby('study_category', observed=True)['exam_score'].mean()
bars = ax2.bar(study_avg.index, study_avg.values,
               color=['#e57373','#ffb74d','#81c784'], edgecolor='white',
               linewidth=0.8, width=0.5)
for bar, val in zip(bars, study_avg.values):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{val:.1f}', ha='center', color=TEXT_COL, fontsize=9, fontweight='bold')
style_ax(ax2, "📚 Study Hours vs Avg Score")
ax2.set_ylabel("Avg Exam Score")
ax2.set_xlabel("Study Category")

# ── Panel 3: Pass / Fail (Donut) ───────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
pf = df['pass_fail'].value_counts()
wedges2, texts2, auto2 = ax3.pie(
    pf, labels=pf.index, autopct='%1.1f%%',
    colors=['#81c784','#e57373'],
    startangle=90, textprops={'color': TEXT_COL, 'fontsize': 9},
    wedgeprops={'edgecolor': '#0f1117', 'linewidth': 2, 'width': 0.6}
)
ax3.set_facecolor(PANEL_BG)
ax3.set_title("✅ Pass / Fail Rate", color=TEXT_COL, fontsize=11, fontweight='bold')

# ── Panel 4: Attendance Band vs Exam Score (Grouped Bar) ───
ax4 = fig.add_subplot(gs[1, 0])
att_avg = df.groupby('attendance_band', observed=True)['exam_score'].mean()
colors4 = ['#e57373','#ffb74d','#4fc3f7','#81c784']
bars4 = ax4.bar(range(len(att_avg)), att_avg.values,
                color=colors4, edgecolor='white', linewidth=0.8, width=0.5)
ax4.set_xticks(range(len(att_avg)))
ax4.set_xticklabels(att_avg.index, rotation=15, ha='right', fontsize=8)
for bar, val in zip(bars4, att_avg.values):
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
             f'{val:.1f}', ha='center', color=TEXT_COL, fontsize=8, fontweight='bold')
style_ax(ax4, "🏫 Attendance Band vs Avg Score")
ax4.set_ylabel("Avg Exam Score")

# ── Panel 5: Correlation Heatmap ───────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
num_df = df[['hours_studied','sleep_hours','attendance_percent','previous_scores','exam_score']]
corr = num_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=ax5, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, linecolor='#0f1117',
            annot_kws={'size':9, 'color': 'white'},
            cbar_kws={'shrink':0.8})
ax5.set_facecolor(PANEL_BG)
ax5.tick_params(colors=TEXT_COL, labelsize=8)
ax5.set_title("🔥 Correlation Heatmap", color=TEXT_COL, fontsize=11, fontweight='bold')

# ── Panel 6: Sleep Quality vs Exam Score (Violin) ──────────
ax6 = fig.add_subplot(gs[1, 2])
sleep_order = ['Poor (<5.5h)', 'Fair (5.5-7.5h)', 'Good (7.5+h)']
sns.violinplot(data=df, x='sleep_quality', y='exam_score',
               order=sleep_order, palette=['#e57373','#ffb74d','#81c784'],
               ax=ax6, linewidth=1.2, inner='quartile')
ax6.set_xticklabels(sleep_order, rotation=12, ha='right', fontsize=8)
style_ax(ax6, "😴 Sleep Quality vs Score")
ax6.set_ylabel("Exam Score")
ax6.set_xlabel("")

# ── Panel 7: Improvement Gap Distribution (Histogram) ──────
ax7 = fig.add_subplot(gs[2, 0])
ax7.hist(df['improvement_gap'], bins=25, color='#4fc3f7',
         edgecolor='#0f1117', linewidth=0.7, alpha=0.9)
ax7.axvline(0, color='#e57373', linestyle='--', linewidth=1.5, label='No Change')
ax7.axvline(df['improvement_gap'].mean(), color='#81c784',
            linestyle='--', linewidth=1.5, label=f"Mean={df['improvement_gap'].mean():.1f}")
ax7.legend(fontsize=8, labelcolor=TEXT_COL,
           facecolor=PANEL_BG, edgecolor=GRID_COL)
style_ax(ax7, "📈 Improvement Gap (Exam − Prev Score)")
ax7.set_xlabel("Score Difference")
ax7.set_ylabel("Count")

# ── Panel 8: Hours Studied vs Exam Score (Scatter) ─────────
ax8 = fig.add_subplot(gs[2, 1])
sc = ax8.scatter(df['hours_studied'], df['exam_score'],
                 c=df['attendance_percent'], cmap='YlOrRd',
                 alpha=0.75, edgecolors='#0f1117', linewidths=0.4, s=55)
cbar = plt.colorbar(sc, ax=ax8)
cbar.set_label('Attendance %', color=TEXT_COL, fontsize=8)
cbar.ax.yaxis.set_tick_params(color=TEXT_COL)
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color=TEXT_COL, fontsize=8)
m, b = np.polyfit(df['hours_studied'], df['exam_score'], 1)
x_line = np.linspace(df['hours_studied'].min(), df['hours_studied'].max(), 100)
ax8.plot(x_line, m*x_line+b, color='#4fc3f7', linewidth=2, label='Trend')
ax8.legend(fontsize=8, labelcolor=TEXT_COL, facecolor=PANEL_BG, edgecolor=GRID_COL)
style_ax(ax8, "🎯 Study Hours vs Score (Attendance Color)")
ax8.set_xlabel("Hours Studied")
ax8.set_ylabel("Exam Score")

# ── Panel 9: Top 10 Students (Horizontal Bar) ──────────────
ax9 = fig.add_subplot(gs[2, 2])
top10 = df.nlargest(10, 'exam_score')[['student_id','exam_score']].sort_values('exam_score')
colors9 = plt.cm.YlGn(np.linspace(0.4, 1.0, 10))
bars9 = ax9.barh(top10['student_id'], top10['exam_score'],
                 color=colors9, edgecolor='white', linewidth=0.6)
for bar, val in zip(bars9, top10['exam_score']):
    ax9.text(val+0.3, bar.get_y()+bar.get_height()/2,
             f'{val}', va='center', color=TEXT_COL, fontsize=8, fontweight='bold')
style_ax(ax9, "🏆 Top 10 Students by Exam Score")
ax9.set_xlabel("Exam Score")

# ── Main Title ──────────────────────────────────────────────
fig.suptitle(
    "🎓 STUDENT PERFORMANCE DASHBOARD",
    fontsize=20, fontweight='bold', color=TEXT_COL, y=0.98
)
plt.savefig("dashboard_jupyter.png", dpi=130, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Dashboard rendered and saved as dashboard_jupyter.png")


In [ ]:
# ── Quick Stats Summary ────────────────────────────────────
print("=" * 52)
print("       📋 DATASET SUMMARY STATISTICS")
print("=" * 52)
print(f"  Total Students        : {len(df)}")
print(f"  Unique Student IDs    : {df['student_id'].nunique()}")
print(f"  Avg Exam Score        : {df['exam_score'].mean():.2f}")
print(f"  Highest Score         : {df['exam_score'].max()} — {df.loc[df['exam_score'].idxmax(),'student_id']}")
print(f"  Lowest Score          : {df['exam_score'].min()} — {df.loc[df['exam_score'].idxmin(),'student_id']}")
print(f"  Pass Rate             : {(df['pass_fail']=='Pass ✅').mean()*100:.1f}%")
print(f"  Avg Study Hours       : {df['hours_studied'].mean():.1f} hrs/day")
print(f"  Avg Sleep Hours       : {df['sleep_hours'].mean():.1f} hrs/day")
print(f"  Avg Attendance        : {df['attendance_percent'].mean():.1f}%")
print(f"  Avg Improvement Gap   : {df['improvement_gap'].mean():.2f} points")
print(f"  Most Common Grade     : {df['performance_grade'].mode()[0]}")
print("=" * 52)


---
## ✅ Notebook Complete
- **Step 1** — Unique data engineered (6 derived columns)
- **Step 2** — 9-panel dark-theme dashboard built with Matplotlib/Seaborn
- **Next** → Open `app.py` in PyCharm for the Streamlit web dashboard
